# Assignment 20
## Building & Deploying a Recommendation System

In [2]:
import pandas as pd
import numpy as np
import re
import string
import nltk

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


nltk.download("stopwords")
print("Libraries Imported Successfully")

Libraries Imported Successfully


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\axita\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
df = pd.read_csv("dataset/movies.csv")


print("Dataset Loaded Successfully")

print()


print("Dataset Shape")
print(df.shape)


print()


print("Column Names")
print(df.columns)


print()


print("First 5 Rows")
df.head()

Dataset Loaded Successfully

Dataset Shape
(4803, 20)

Column Names
Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

First 5 Rows


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [4]:
print("Dataset Information")

df.info()

print("Missing Values")

print(df.isnull().sum())

Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  s

# Task 1
## Load & Understand Dataset

In [5]:
print("Text Columns Available")
print()

print("Title Column:")
print("title")


print("Text Feature Column:")
print("overview")


print()
print("Sample Movie Overview")

print(df["overview"].head())

Text Columns Available

Title Column:
title
Text Feature Column:
overview

Sample Movie Overview
0    In the 22nd century, a paraplegic Marine is di...
1    Captain Barbossa, long believed to be dead, ha...
2    A cryptic message from Bond’s past sends him o...
3    Following the death of District Attorney Harve...
4    John Carter is a war-weary, former military ca...
Name: overview, dtype: object


# Task 2 
## Text Preprocessing

In [6]:
stop_words = set(stopwords.words("english"))


df["overview"] = df["overview"].fillna("")

def clean_text(text):
    text = text.lower()
    text = text.translate(
        str.maketrans("","",string.punctuation)
    )


    text = re.sub("[^a-zA-Z]"," ",text)

    words = text.split()
    cleaned_words = []

    for word in words:
        if word not in stop_words:
            cleaned_words.append(word)

    return " ".join(cleaned_words)



df["clean_text"] = df["overview"].apply(clean_text)

print("Text Cleaning Completed")

print()


df[["title","clean_text"]].head()

Text Cleaning Completed



,title,clean_text
0,Avatar,nd century paraplegic marine dispatched moon p...
1,Pirates of the Caribbean: At World's End,captain barbossa long believed dead come back ...
2,Spectre,cryptic message bond past sends trail uncover ...
3,The Dark Knight Rises,following death district attorney harvey dent ...
4,John Carter,john carter warweary former military captain w...


In [7]:
print("Original Text")
print(df["overview"].head(3))

print()

print("Cleaned Text")
print(df["clean_text"].head(3))

Original Text
0    In the 22nd century, a paraplegic Marine is di...
1    Captain Barbossa, long believed to be dead, ha...
2    A cryptic message from Bond’s past sends him o...
Name: overview, dtype: object

Cleaned Text
0    nd century paraplegic marine dispatched moon p...
1    captain barbossa long believed dead come back ...
2    cryptic message bond past sends trail uncover ...
Name: clean_text, dtype: object


# PART 2
## Task 3: TF-IDF Vectorization

In [8]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)


tfidf_matrix = tfidf.fit_transform(df["clean_text"])


print("TF-IDF Vectorization Completed")

print()

print("TF-IDF Matrix Shape")
print(tfidf_matrix.shape)

TF-IDF Vectorization Completed

TF-IDF Matrix Shape
(4803, 5000)


In [9]:
print("Sample TF-IDF Features")

print()

features = tfidf.get_feature_names_out()

print(features[:20])

print()

print("Total Features")

print(len(features))

Sample TF-IDF Features

['aaron' 'abandoned' 'abandons' 'abducted' 'abilities' 'ability' 'able'
 'aboard' 'abroad' 'absence' 'abuse' 'abused' 'abusive' 'academic'
 'academy' 'academy award' 'accept' 'accepted' 'accepts' 'access']

Total Features
5000


# Task 4
## Similarity Computation

In [10]:
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity Matrix Created Successfully")

print()

print("Similarity Matrix Shape")

print(similarity_matrix.shape)

print("Sample Similarity Scores")

print()

print(similarity_matrix[0][:10])

Similarity Matrix Created Successfully

Similarity Matrix Shape
(4803, 4803)
Sample Similarity Scores

[1.         0.         0.         0.03346604 0.02096321 0.04333675
 0.         0.05118473 0.         0.        ]


# PART 3 Recommendation Logic
## Task 5
## Build Recommendation Function

In [11]:
def recommend(movie_name, top_n=5):
    if movie_name not in list(df["title"].values):
        return "Movie not found"


    movie_index = df[df["title"] == movie_name].index[0]


    similarity_scores = list(
        enumerate(
            similarity_matrix[movie_index]
        )
    )


    similarity_scores = sorted(
        similarity_scores,
        key=lambda x:x[1],
        reverse=True
    )


    recommended_movies = []


    for i in similarity_scores[1:top_n+1]:
        recommended_movies.append(
            df.iloc[i[0]]["title"]
        )

    return recommended_movies

In [12]:
print("Testing Recommendation Function")

print()

movie = df["title"].iloc[0]

print("Selected Movie:")

print(movie)

print()

print("Recommended Movies:")

recommend(movie,5)

Testing Recommendation Function

Selected Movie:
Avatar

Recommended Movies:


['Apollo 18',
 'The American',
 'The Inhabited Island',
 'Tears of the Sun',
 'The Adventures of Pluto Nash']

In [13]:
print("Testing with Different Movies")

print()

test_movies = [
    df["title"].iloc[10],
    df["title"].iloc[50],
    df["title"].iloc[100]
]


for movie in test_movies:
    print("Movie:")
    print(movie)
    print("Recommendations:")

    print(recommend(movie,5))
    
    print()


Testing with Different Movies

Movie:
Superman Returns
Recommendations:
['Superman IV: The Quest for Peace', 'Superman II', 'Superman', 'Superman III', 'The Crow']

Movie:
Prince of Persia: The Sands of Time
Recommendations:
['Tusk', 'The Mortal Instruments: City of Bones', 'Robin Hood: Prince of Thieves', 'Night Watch', 'Once Upon a Time in the West']

Movie:
The Curious Case of Benjamin Button
Recommendations:
['The Believer', 'Lovely, Still', 'Opal Dream', 'The Patriot', 'Nowhere Boy']



In [14]:
print("Task 5 Completed Successfully")
print()

print("Observation")

print()

print("The recommendation system finds movies with similar descriptions.")

print("TF-IDF converts movie descriptions into numerical vectors.")

print("Cosine similarity compares these vectors to find similar items.")

Task 5 Completed Successfully

Observation

The recommendation system finds movies with similar descriptions.
TF-IDF converts movie descriptions into numerical vectors.
Cosine similarity compares these vectors to find similar items.


# Saving Files for Streamlit Application

The Streamlit application requires:

- Movie data
- TF-IDF vectorizer
- TF-IDF matrix
- Similarity matrix

These files are saved so the application can load them directly.

In [15]:
import pickle

# Saving dataframe

df.to_pickle("movies_data.pkl")


# Saving similarity matrix

with open("similarity.pkl","wb") as file:
    pickle.dump(
        similarity_matrix,file
)


# Saving vectorizer

with open("tfidf_vectorizer.pkl","wb") as file:
    pickle.dump(tfidf,file)


print("Files Saved Successfully")

Files Saved Successfully


# Final Observation

The content-based recommendation system was successfully created.

Steps completed:

- Dataset loading
- Text preprocessing
- TF-IDF feature extraction
- Cosine similarity calculation
- Recommendation function creation
- Testing recommendations
- Saving files for deployment

In [16]:
print("Assignment 20 Notebook Completed Successfully")

print()

print("Content Based Movie Recommendation System is Ready for Deployment.")

Assignment 20 Notebook Completed Successfully

Content Based Movie Recommendation System is Ready for Deployment.
